# Jak komputer czyta tekst - od liczenia słów do wektorów

Ten notatnik to companion do blog posta: [Jak komputer czyta tekst](jak-komputer-czyta-tekst.html)

**Instalacja:**
```
pip install tiktoken transformers tokenizers scikit-learn pandas gensim numpy
```

---
## 1. Tokenizacja

### Tiktoken - jak GPT tnie tekst

In [ ]:
import tiktoken

text = "Nieprawdopodobnie szczęśliwy"

gpt2 = tiktoken.get_encoding("gpt2")
gpt4 = tiktoken.get_encoding("cl100k_base")

print("GPT-2:", gpt2.encode(text))
print("GPT-4:", gpt4.encode(text))

print("GPT-2:", [gpt2.decode([t]) for t in gpt2.encode(text)])
print("GPT-4:", [gpt4.decode([t]) for t in gpt4.encode(text)])

### Polskie tokenizatory

In [ ]:
from transformers import AutoTokenizer

text = "Nieprawdopodobnie szczęśliwy"

herbert = AutoTokenizer.from_pretrained("allegro/herbert-base-cased")
print("HerBERT (Allegro, polski WordPiece):", herbert.tokenize(text))

roberta = AutoTokenizer.from_pretrained("sdadas/polish-roberta-base-v2")
print("Polish RoBERTa (polski SentencePiece):", roberta.tokenize(text))

### Własny tokenizer BPE

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(vocab_size=300, special_tokens=["[UNK]"])

corpus = [
    "Ala ma kota i psa",
    "Kot siedzi na macie",
    "Pies biega po parku",
    "Ala ma kota i kota i jeszcze raz kota",
    "Kot lubi mleko i kot lubi spać",
    "Pies lubi biegać po parku i gonić kota",
    "Szczęśliwy kot to kot który ma dużo mleka",
    "Nieprawdopodobnie szczęśliwy pies biega po parku",
    "Szczęśliwy Ala ma kota i psa i szczęśliwy dom",
    "Dom to miejsce gdzie mieszka szczęśliwa rodzina",
]

tokenizer.train_from_iterator(corpus, trainer)
print(f"Rozmiar słownika: {tokenizer.get_vocab_size()}")

test = "Nieprawdopodobnie szczęśliwy"
encoding = tokenizer.encode(test)
print(f'"{test}" → {encoding.tokens}')

---
## 2. BoW i TF-IDF

### Bag of Words (BoW)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

corpus = [
    "Ten film jest świetny",
    "Ten film jest beznadziejny",
    "Świetny film polecam",
]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)

df = pd.DataFrame(
    X.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=["Dok 1", "Dok 2", "Dok 3"],
)
print(df)

### TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

corpus = [
    "Ten film jest świetny",
    "Ten film jest beznadziejny",
    "Świetny film polecam",
]

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=["Dok 1", "Dok 2", "Dok 3"],
)
print(df.round(2))

### TF-IDF jako wyszukiwarka

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

corpus = [
    "Ten film jest świetny",
    "Ten film jest beznadziejny",
    "Świetny film polecam",
]

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

query = "świetny film"
query_vec = vectorizer.transform([query])

similarities = cosine_similarity(query_vec, tfidf_matrix)[0]
ranked = sorted(zip(corpus, similarities), key=lambda x: -x[1])
for i, (doc, sim) in enumerate(ranked, 1):
    print(f'{i}. "{doc}" → {sim:.3f}')

### N-gramy z TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

corpus = [
    "Ten film jest świetny",
    "Ten film jest beznadziejny",
    "Świetny film polecam",
]

vectorizer = TfidfVectorizer(ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(corpus)

df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=["Dok 1", "Dok 2", "Dok 3"],
)
print(df.round(2))

---
## 3. Łańcuchy Markowa - generator tekstu

In [ ]:
import random

corpus = "ala ma kota ala ma psa kot lubi mleko ala lubi kota"
tokens = corpus.split()

trigrams = {}
for i in range(len(tokens) - 3):
    key = (tokens[i], tokens[i + 1], tokens[i + 2])
    next_word = tokens[i + 3]
    if key not in trigrams:
        trigrams[key] = []
    trigrams[key].append(next_word)

current = ("ala", "ma", "kota")  # to jest nasz "prompt"!
output = list(current)

for _ in range(10):
    if tuple(output[-3:]) not in trigrams:
        break
    possibilities = trigrams[tuple(output[-3:])]
    output.append(random.choice(possibilities))

print(" ".join(output))

---
## 4. Naive Bayes - klasyfikator spamu

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer

emails = [
    "Kup viagra tanio teraz",
    "Viagra darmowa oferta",
    "Spotkanie jutro o 10",
    "Prześlij mi raport jutro",
    "Tanie leki online kup teraz",
    "Raport z wczoraj prześlij",
]
labels = [1, 1, 0, 0, 1, 0]  # 1=spam, 0=nie-spam

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(emails)

classifier = MultinomialNB(alpha=1.0)
classifier.fit(X, labels)

new_email = vectorizer.transform(["Viagra jutro spotkanie"])
prediction = classifier.predict(new_email)
print("Spam!" if prediction[0] == 1 else "Nie-spam.")

---
## 5. Word2Vec - embeddingi od zera

### CBOW od zera (czysty numpy)

In [ ]:
import numpy as np

np.random.seed(42)

# --- 1. Słownik i one-hot encoding ---
corpus = "kot siedzi na macie i śpi pies siedzi na dywanie i śpi kot biega po pokoju pies biega po parku".split()
vocab = list(set(corpus))
word2idx = {w: i for i, w in enumerate(vocab)}
vocab_size = len(vocab)

def one_hot(word):
    vec = np.zeros(vocab_size)
    vec[word2idx[word]] = 1
    return vec

print(f'Słownik ({vocab_size} słów): {vocab}')
print(f'One-hot "kot": {one_hot("kot")}')

In [ ]:
# --- 2. Macierz wag (to będą nasze embeddingi) ---
embed_dim = 5  # w prawdziwym Word2Vec to 100-300
W1 = np.random.randn(vocab_size, embed_dim) * 0.01  # słownik → embedding
W2 = np.random.randn(embed_dim, vocab_size) * 0.01  # embedding → słownik

# --- 3. Pary treningowe (CBOW: kontekst → środek) ---
window = 2
pairs = []
for i in range(window, len(corpus) - window):
    context = corpus[i - window:i] + corpus[i + 1:i + window + 1]
    target = corpus[i]
    pairs.append((context, target))

print(f'Liczba par treningowych: {len(pairs)}')
print(f'Przykładowa para: kontekst={pairs[0][0]} → cel={pairs[0][1]}')

In [ ]:
# --- 4. Trening (gradient descent) ---
def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

lr = 0.05
for epoch in range(500):
    loss = 0
    for context_words, target_word in pairs:
        context_idx = [word2idx[w] for w in context_words]
        target_idx = word2idx[target_word]

        # forward: uśrednione embeddingi kontekstu → przewidujemy środek
        hidden = np.mean(W1[context_idx], axis=0)
        output = softmax(hidden @ W2)
        loss -= np.log(output[target_idx] + 1e-8)

        # backward: aktualizujemy wagi
        grad = output.copy()
        grad[target_idx] -= 1
        W2 -= lr * np.outer(hidden, grad)
        grad_hidden = grad @ W2.T
        for idx in context_idx:
            W1[idx] -= lr * grad_hidden / len(context_idx)

    if (epoch + 1) % 100 == 0:
        print(f'Epoch {epoch+1}, loss: {loss:.3f}')

In [ ]:
# --- 5. Wynik: embeddingi! ---
for word in ["kot", "pies", "siedzi", "biega"]:
    print(f"{word}: {W1[word2idx[word]].round(3)}")

print()

# --- 6. Podobieństwo kosinusowe ---
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

print(f'kot ↔ pies: {cosine(W1[word2idx["kot"]], W1[word2idx["pies"]]):.3f}')
print(f'kot ↔ siedzi: {cosine(W1[word2idx["kot"]], W1[word2idx["siedzi"]]):.3f}')
print(f'kot ↔ biega: {cosine(W1[word2idx["kot"]], W1[word2idx["biega"]]):.3f}')

### Word2Vec z gensim (gotowy model)

In [ ]:
from gensim.models import Word2Vec

corpus = [
    ["kot", "siedzi", "na", "macie", "i", "śpi"],
    ["pies", "siedzi", "na", "dywanie", "i", "śpi"],
    ["kot", "biega", "po", "pokoju"],
    ["pies", "biega", "po", "parku"],
    ["kot", "i", "pies", "bawią", "się", "w", "ogrodzie"],
    ["kot", "je", "karmę", "z", "miski"],
    ["pies", "je", "karmę", "z", "miski"],
    ["kot", "łapie", "mysz", "w", "domu"],
    ["pies", "goni", "kota", "po", "ogrodzie"],
    ["ptak", "siedzi", "na", "gałęzi", "drzewa"],
    ["ryba", "pływa", "w", "akwarium"],
    ["samochód", "jeździ", "po", "drodze"],
    ["rower", "jeździ", "po", "ścieżce"],
    ["kot", "mruczy", "gdy", "głaszczesz", "go"],
    ["pies", "merda", "ogonem", "gdy", "widzi", "pana"],
]

model = Word2Vec(
    sentences=corpus,
    vector_size=10,   # wymiar wektora (mały = edukacyjny)
    window=3,         # rozmiar okna kontekstowego
    min_count=1,      # ignoruj słowa rzadsze niż N
    sg=0,             # 0 = CBOW, 1 = Skip-gram
    epochs=200,       # ile przejść po danych
)

print(model.wv.most_similar("kot", topn=3))
print(model.wv.similarity("kot", "pies"))
print(model.wv.similarity("kot", "samochód"))

### Word2Vec + łańcuchy Markowa = generator z embeddingami

In [ ]:
from gensim.models import Word2Vec
import numpy as np
from collections import defaultdict
import random

corpus = [
    "kot siedzi na macie i śpi",
    "pies siedzi na dywanie i śpi",
    "kot biega po pokoju",
    "pies biega po parku",
    "kot i pies bawią się w ogrodzie",
    "kot je karmę z miski",
    "pies je karmę z miski",
    "kot łapie mysz w domu",
    "pies goni kota po ogrodzie",
    "ptak siedzi na gałęzi drzewa",
    "kot patrzy przez okno i mruczy",
    "pies szczeka na listonosza",
    "kot śpi cały dzień na kanapie",
]

tokenized = [sentence.split() for sentence in corpus]

w2v_model = Word2Vec(sentences=tokenized, vector_size=10, window=3, min_count=1, epochs=200)

# budujemy macierz przejść (jak w łańcuchach Markowa)
transitions = defaultdict(list)
for sentence in tokenized:
    for i in range(len(sentence) - 1):
        transitions[sentence[i]].append(sentence[i + 1])

def generate(start, length=6):
    words = [start]
    for _ in range(length):
        current = words[-1]
        if current not in transitions:
            break

        candidates = transitions[current]

        # uśredniony wektor ostatnich słów = kontekst
        context_vector = np.mean(
            [w2v_model.wv[w] for w in words[-3:] if w in w2v_model.wv],
            axis=0,
        )

        # punktujemy kandydatów za podobieństwo do kontekstu
        scored = []
        for candidate in candidates:
            if candidate in w2v_model.wv:
                similarity = np.dot(context_vector, w2v_model.wv[candidate]) / (
                    np.linalg.norm(context_vector) * np.linalg.norm(w2v_model.wv[candidate]) + 1e-8
                )
                scored.append((candidate, similarity))

        if scored:
            # losujemy, ale z wagami — podobne słowa mają większą szansę
            weights = [score + 1 for _, score in scored]
            chosen = random.choices([w for w, _ in scored], weights=weights, k=1)[0]
            words.append(chosen)
        else:
            words.append(random.choice(candidates))
    return " ".join(words)

for _ in range(5):
    print(generate("kot"))